# 第 1 周末练习 —— 技术问答解释器（本地 Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：面向新手的详细解释
- **额外要求**：用**流式（streaming）**一边生成一边 `print`，最后再用 Markdown 展示全文

这是你在课程期间自己也能天天用的工具。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定角色，user 放具体问题 |
| 流式输出 `stream=True` | `for chunk in response` 边收边打印 |
| Ollama 本地模型 | `base_url=http://localhost:11434/v1`，模型名 `llama3.2` |

## 怎么跑

1. 确保 Ollama 在跑，且已 `ollama pull llama3.2`
2. 从上到下运行单元格；可在「提问」格改写 `question`


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 dotenv 导入 load_dotenv：把 .env 读进环境变量（本练习客户端密钥写死为 "ollama"，仍保留加载习惯）
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：对接 Ollama 的 OpenAI 兼容 /v1 接口
from openai import OpenAI
# 从 IPython.display 导入 Markdown 与 display：最后把完整回答渲染成漂亮 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 常量：本地模型名集中配置 ==========

# 本地 Ollama 模型名：需与本机 `ollama list` 中的名字一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境 + 客户端：指向本机 Ollama ==========

# 加载 .env（即使本格主要用固定 base_url，也保持与课程一致的加载步骤）
load_dotenv()
# 创建 OpenAI 客户端，但 base_url 指向本地 Ollama 的 OpenAI 兼容端点
# api_key="ollama" 是占位：本地 Ollama 通常不校验密钥，但 SDK 仍要求传一个非空字符串
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 调用本地 Llama 3.2：流式打印 + 最终 Markdown 展示 ==========

# system prompt 保留英文：规定「技术专家、擅长帮新手」的回答风格
system_prompt = "You are a technical expert proficient in all programming languages and technical frameworks, and are particularly skilled at helping novice programmers solve problems."
# 把具体问题拼进 user prompt（f-string）；指令正文保持英文
user_prompt = f"Please give a detailed explanation to the following question: {question}"
# messages：Chat Completions 的标准结构 —— system 定角色，user 放任务
messages = [
    {"role": "system", "content":system_prompt },
    {"role": "user", "content": user_prompt}
]
# 发起流式请求：model 用常量；stream=True 表示服务端持续推送 token/块
response = openai.chat.completions.create(model=MODEL_LLAMA, messages=messages,stream=True)
# 累加完整文本，供最后一次 Markdown 渲染
full_content = ""
# 遍历每个流式 chunk
for chunk in response:
    # 有的 chunk 没有 content（例如 role 信息），要判空再拼接
    if chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        full_content += content
        # end="" 不换行；flush=True 立刻刷到屏幕，实现打字机效果
        print(content, end="", flush=True)
# 流结束后，用 Markdown 再展示一遍完整回答（便于笔记本里排版阅读）
display(Markdown(full_content))
